<a href="https://colab.research.google.com/github/saitejamudapalli/Project-HealthCare-Provider-Analysis/blob/main/DIMENSIONAL_TABLES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#CODE FOR CREATING DIMENSION TABLES

# ===========================================
# 🧩 Create Dimension Tables in BigQuery (Fully Fixed)
# ===========================================

from google.cloud import bigquery
from google.oauth2 import service_account
import pandas as pd
import numpy as np

# --- 🔐 Setup ---
key_path = "/content/even-blueprint-441418-p2-043f8a9d855b.json(KEY).json"   # 👈 replace with your JSON key path
project_id = "even-blueprint-441418-p2"                   # 👈 replace with your project ID
dataset_id = "SILVER_LAYER"                        # 👈 replace with your dataset name

# BigQuery client
creds = service_account.Credentials.from_service_account_file(key_path)
client = bigquery.Client(credentials=creds, project=project_id)

# --- 📦 Load CSV ---
csv_path = "/content/PATIENTS_SILVER.csv"        # 👈 replace with your CSV path
df = pd.read_csv(csv_path, low_memory=False)

# ✅ Convert date columns safely
for col in ["birthdate", "deathdate", "_ingest_time_utc"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# ✅ Convert ZIP code to string (avoid ArrowTypeError)
if "zip" in df.columns:
    df["zip"] = df["zip"].astype(str).str.replace(r"\.0$", "", regex=True)

# ✅ Convert lat/lon safely to numeric
for col in ["lat", "lon"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# ===========================================
# ✅ DIM_PATIENT (Completely Clean — No NULLs in Any Column)
# ===========================================
dim_patient = df[[
    "id", "first", "last", "gender", "birthdate", "deathdate", "marital",
    "race", "ethnicity", "ssn", "drivers", "passport", "prefix", "suffix", "maiden"
]].rename(columns={
    "id": "patient_id",
    "first": "first_name",
    "last": "last_name",
    "marital": "marital_status"
})

# 🚫 Drop ALL rows that contain ANY null or NaN value across ALL columns
dim_patient = dim_patient.dropna(how="any").reset_index(drop=True)

# 🧹 Optionally: remove empty strings or spaces pretending to be data
dim_patient = dim_patient.replace(r'^\s*$', pd.NA, regex=True).dropna(how="any")

# ✅ Convert datatypes (BigQuery likes clean types)
dim_patient["birthdate"] = pd.to_datetime(dim_patient["birthdate"], errors="coerce").dt.date
dim_patient["deathdate"] = pd.to_datetime(dim_patient["deathdate"], errors="coerce").dt.date

# Drop again any rows that became null after conversion
dim_patient = dim_patient.dropna(how="any").reset_index(drop=True)

# --- BigQuery Load ---
table_id = f"{project_id}.{dataset_id}.dim_patient"
client.delete_table(table_id, not_found_ok=True)

schema = [
    bigquery.SchemaField("patient_id", "STRING"),
    bigquery.SchemaField("first_name", "STRING"),
    bigquery.SchemaField("last_name", "STRING"),
    bigquery.SchemaField("gender", "STRING"),
    bigquery.SchemaField("birthdate", "DATE"),
    bigquery.SchemaField("deathdate", "DATE"),
    bigquery.SchemaField("marital_status", "STRING"),
    bigquery.SchemaField("race", "STRING"),
    bigquery.SchemaField("ethnicity", "STRING"),
    bigquery.SchemaField("ssn", "STRING"),
    bigquery.SchemaField("drivers", "STRING"),
    bigquery.SchemaField("passport", "STRING"),
    bigquery.SchemaField("prefix", "STRING"),
    bigquery.SchemaField("suffix", "STRING"),
    bigquery.SchemaField("maiden", "STRING"),
]

client.create_table(bigquery.Table(table_id, schema=schema))
client.load_table_from_dataframe(dim_patient, table_id).result()

print(f"✅ dim_patient created and loaded successfully — {len(dim_patient)} clean rows (no nulls).")


# ===========================================
# ✅ DIM_LOCATION (Completely Clean — No NULL / NaN / Empty Values)
# ===========================================

# Create dimension table from relevant columns
dim_location = df[["address", "city", "state", "county", "zip", "birthplace"]].copy()

# ✅ Convert all values to string (prevents dtype issues)
dim_location = dim_location.astype(str)

# ✅ Replace string forms of nulls / blanks with real NaN
dim_location = dim_location.replace(
    to_replace=["nan", "NaN", "None", "NULL", "null", ""],
    value=pd.NA
)

# ✅ Drop ALL rows that contain ANY null or blank value
dim_location = dim_location.dropna(how="any").reset_index(drop=True)

# ✅ Remove duplicates
dim_location = dim_location.drop_duplicates().reset_index(drop=True)

# ✅ Add surrogate key
dim_location["location_id"] = range(1, len(dim_location) + 1)

# --- BigQuery Load ---
table_id = f"{project_id}.{dataset_id}.dim_location"
client.delete_table(table_id, not_found_ok=True)

schema = [
    bigquery.SchemaField("location_id", "INTEGER"),
    bigquery.SchemaField("address", "STRING"),
    bigquery.SchemaField("city", "STRING"),
    bigquery.SchemaField("state", "STRING"),
    bigquery.SchemaField("county", "STRING"),
    bigquery.SchemaField("zip", "STRING"),
    bigquery.SchemaField("birthplace", "STRING"),
]

client.create_table(bigquery.Table(table_id, schema=schema))
client.load_table_from_dataframe(dim_location, table_id).result()

print(f"✅ dim_location created and loaded successfully — {len(dim_location)} clean rows (no NaN / NULL / blanks).")


# ===========================================
# ✅ DIM_TIME (Completely Clean — No NULL Values)
# ===========================================
dim_time = pd.DataFrame()
dim_time["time_id"] = range(1, len(df) + 1)
dim_time["birthdate"] = pd.to_datetime(df["birthdate"], errors="coerce")
dim_time["deathdate"] = pd.to_datetime(df["deathdate"], errors="coerce")
dim_time["record_ingest_time"] = pd.to_datetime(df.get("_ingest_time_utc", pd.NaT), errors="coerce")

# Extract date parts
dim_time["birth_year"] = dim_time["birthdate"].dt.year
dim_time["birth_month"] = dim_time["birthdate"].dt.month
dim_time["birth_day"] = dim_time["birthdate"].dt.day
dim_time["death_year"] = dim_time["deathdate"].dt.year
dim_time["death_month"] = dim_time["deathdate"].dt.month
dim_time["death_day"] = dim_time["deathdate"].dt.day

# 🚫 Replace empty strings or spaces with NaN before cleaning
dim_time = dim_time.replace(r'^\s*$', pd.NA, regex=True)

# 🚫 Drop ALL rows containing ANY null value in ANY column
dim_time = dim_time.dropna(how="any").reset_index(drop=True)

# ✅ Recreate sequential time_id
dim_time["time_id"] = range(1, len(dim_time) + 1)

# --- BigQuery Load ---
table_id = f"{project_id}.{dataset_id}.dim_time"
client.delete_table(table_id, not_found_ok=True)

schema = [
    bigquery.SchemaField("time_id", "INTEGER"),
    bigquery.SchemaField("birthdate", "DATE"),
    bigquery.SchemaField("deathdate", "DATE"),
    bigquery.SchemaField("record_ingest_time", "TIMESTAMP"),
    bigquery.SchemaField("birth_year", "INTEGER"),
    bigquery.SchemaField("birth_month", "INTEGER"),
    bigquery.SchemaField("birth_day", "INTEGER"),
    bigquery.SchemaField("death_year", "INTEGER"),
    bigquery.SchemaField("death_month", "INTEGER"),
    bigquery.SchemaField("death_day", "INTEGER"),
]

client.create_table(bigquery.Table(table_id, schema=schema))
client.load_table_from_dataframe(dim_time, table_id).result()

print(f"✅ dim_time created and loaded successfully — {len(dim_time)} clean rows (no nulls).")




print("\n🎉 All dimension tables created successfully and uploaded to BigQuery without datatype errors!")
